In [ ]:
!pip install libero
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
!pip3 install diffusers scikit-image scikit-video "zarr>=3" numcodecs
!pip install transformers

In [ ]:
# dependencies for headless rendering in Colab
!apt-get update -qq
!apt-get install -y xvfb python3-opengl -qq

In [ ]:
!git clone https://github.com/GoncaloMark/DuPLO-VLA.git

In [ ]:
%cd DuPLO-VLA
!git switch v2

In [ ]:
!git pull

In [ ]:
import torch
import os
import os
import collections
import numpy as np
import torch
from skvideo.io import vwrite
from IPython.display import Video, display
from tqdm.auto import tqdm

from google.colab import userdata
from huggingface_hub import login, hf_hub_download
import sys, os, copy

# diffusion policy import
from typing import Tuple, Sequence, Dict, Union, Optional, Callable
import numpy as np
import math
import torch
import torch.nn as nn
import torchvision
import collections
import zarr
from diffusers.schedulers.scheduling_ddpm import DDPMScheduler
from diffusers.training_utils import EMAModel
from diffusers.optimization import get_scheduler
from tqdm.auto import tqdm

# env import
import gym
from gym import spaces
import cv2
import skimage.transform as st
from skvideo.io import vwrite
from IPython.display import Video

import math

sys.path.append('/content/DuPLO-VLA/src')
os.chdir('/content/DuPLO-VLA')

import torch.nn.functional as F

from vlm.vlm import VisualTaskPlanner
from action_heads.diffusion import RDTPolicy

try:
    from libero.libero.envs import OffScreenRenderEnv
    from libero.libero.benchmark import get_benchmark
    from libero.libero import get_libero_path
except ImportError:
    print("Please install the LIBERO benchmark to run this environment.")

In [ ]:
os.environ['MUJOCO_GL'] = 'egl'

print("Headless rendering configured.")

In [ ]:
planner = VisualTaskPlanner(
    num_pooling_queries = 64,
    latent_dim          = 512,
    num_sampled_layers  = 4,
    q_hidden_dim        = 768,
    num_attention_heads = 8,
    dropout=0.1,
)

In [ ]:
device = torch.device("cuda")

obs_horizon = 1
proprio_horizon = 4
pred_horizon = 16
action_horizon = 8
action_dim = 7
proprio_dim = 8
state_dim = proprio_horizon * proprio_dim

vlm_num_layers = 4
vlm_hidden_dim = 2560
layer_indices = [4, 12, 25, 34]

num_queries = 64
q_hidden_dim = 768
latent_dim = 512
encoder_dropout = 0.1

num_train_timesteps = 1000
num_inference_timesteps = 20
prediction_type = "sample"
beta_schedule = "squaredcos_cap_v2"

num_epochs = 120
lr = 2e-4
encoder_lr_mult = 0.3
weight_decay = 1e-6
grad_clip_norm = 1.0
warmup_steps = 500
batch_size = 256

AUX_WEIGHT = 0.5


In [ ]:
policy = RDTPolicy(
    action_dim                = action_dim,
    horizon                   = pred_horizon,
    state_dim                 = state_dim,
    cond_dims                 = {"latent": latent_dim},
    cond_seq_lens             = {},       # Q-Pooler queries -> no pos embed
    hidden_dim                = 512,
    depth                     = 6,
    num_heads                 = 8,
    num_train_timesteps       = num_train_timesteps,
    num_inference_timesteps   = num_inference_timesteps,
    prediction_type           = prediction_type,
    beta_schedule             = beta_schedule,
).to(device)

In [ ]:
checkpoint = torch.load("/content/duplo_libero_v2_epoch_120.pt", map_location="cpu", weights_only=False)

policy.load_pretrained(checkpoint, key="ema_policy")
policy.eval()

In [ ]:
checkpoint = torch.load("/content/duplo_libero_v2_epoch_120.pt", map_location="cpu", weights_only=False)

planner.load_encoder_weights(
    checkpoint,
    key="ema_encoder",
)
planner.eval()

In [ ]:
policy.to("cuda").to(torch.bfloat16)
policy.eval()
planner.task_encoder.to("cuda")
planner.eval()
policy_device = next(policy.parameters()).device

In [ ]:
stats = checkpoint["stats"]

In [ ]:
trained_instructions = set([
    "open the middle drawer of the cabinet",
    "put the bowl on the stove",
    "put the wine bottle on top of the cabinet",
    "open the top drawer and put the bowl inside",
    "put the bowl on top of the cabinet",
    "push the plate to the front of the stove",
    "put the cream cheese in the bowl",
    "turn on the stove",
    "put the bowl on the plate",
    "put the wine bottle on the rack",
])

all_tasks = []
for bench_name in ["libero_goal"]: #, "libero_10", "libero_spatial", "libero_object"]:
    try:
        benchmark = get_benchmark(bench_name)(0)
        for i in range(benchmark.get_num_tasks()):
            task = benchmark.get_task(i)
            init_states = benchmark.get_task_init_states(i)
            all_tasks.append({
                "suite":       bench_name,
                "task_id":     i,
                "bddl_file":   task.bddl_file,
                "language":    task.language,
                "init_states": init_states,
            })
    except Exception as e:
        print(f"Skip {bench_name}: {e}")


In [ ]:
def to_hwc_uint8_float(img):
    if isinstance(img, torch.Tensor):
        img = img.cpu().numpy()
    if img.ndim != 3:
        raise ValueError(f"Expected 3D image, got {img.shape}")
    if img.shape[0] == 3 and img.shape[-1] != 3:
        img = np.transpose(img, (1, 2, 0))
    img = img.astype(np.float32)
    img = np.flipud(img).copy() # match LeRobot orientation
    return img


@torch.no_grad()
def extract_latent_seq(planner, main_img_hwc, wrist_img_hwc, instruction):
    all_h, mask = planner.extract_features_batch(
        [main_img_hwc], [wrist_img_hwc], [instruction]
    )
    sampled = torch.stack([all_h[i] for i in layer_indices], dim=1)
    # sampled: (1, num_layers, L, D); repeat across obs_horizon
    sampled = sampled.unsqueeze(1).expand(-1, obs_horizon, -1, -1, -1)
    mask = mask.unsqueeze(1).expand(-1, obs_horizon, -1)

    enc_out = planner.task_encoder(
        vlm_hidden_states=sampled.to(torch.bfloat16),
        key_padding_mask=mask,
    )
    return enc_out["latent_seq"]


def normalize_state(s, stats):
    smin = stats["agent_pos"]["min"]
    smax = stats["agent_pos"]["max"]
    s = (s - smin) / (smax - smin)
    return s * 2 - 1


def unnormalize_action(a, stats):
    amin = stats["action"]["min"]
    amax = stats["action"]["max"]
    a = (a + 1) / 2
    return a * (amax - amin) + amin


In [ ]:
from scipy.spatial.transform import Rotation as R

def get_state_from_obs(obs, proprio_dim=8):
    eef_pos = obs["robot0_eef_pos"]
    eef_quat = obs["robot0_eef_quat"]
    gripper = obs["robot0_gripper_qpos"]

    eef_euler = R.from_quat(eef_quat).as_euler("xyz", degrees=False)

    if eef_euler[0] < 0:
        eef_euler[0] += 2 * np.pi

    state = np.concatenate([eef_pos, eef_euler, gripper]).astype(np.float32)
    return state[:proprio_dim]

In [ ]:
import os
import collections
import numpy as np
import torch
from skvideo.io import vwrite

video_dir = "/content/rollouts"
os.makedirs(video_dir, exist_ok=True)


def run_rollout(env, init_state, instruction, max_steps=400):
    obs = env.reset()
    env.set_init_state(init_state)

    for _ in range(5):
        obs, _, _, _ = env.step(np.zeros(action_dim))

    state_buffer = collections.deque(maxlen=proprio_horizon)
    for _ in range(proprio_horizon):
        state_buffer.append(get_state_from_obs(obs, proprio_dim))

    frames = []
    success = False
    step_i = 0
    pending_actions = []

    while step_i < max_steps:
        if not pending_actions:
            main_img  = to_hwc_uint8_float(obs["agentview_image"])
            wrist_img = to_hwc_uint8_float(obs["robot0_eye_in_hand_image"])

            latent_seq = extract_latent_seq(planner, main_img, wrist_img, instruction)

            s_np = np.stack(list(state_buffer))
            s_np = normalize_state(s_np, stats)
            s = torch.from_numpy(s_np).to(policy_device).to(torch.bfloat16)
            s = s.reshape(1, -1)

            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                pred = policy.sample(
                    state=s,
                    cond_dict={"latent": latent_seq},
                )
            pred = pred.squeeze(0).float().cpu().numpy()
            pred = unnormalize_action(pred, stats)
            pending_actions = list(pred[:action_horizon])

        a = pending_actions.pop(0)
        obs, _, done, info = env.step(a)
        state_buffer.append(get_state_from_obs(obs, proprio_dim))

        frames.append(np.flipud(obs["agentview_image"]).copy())
        step_i += 1

        if env.check_success():
            success = True
            break

    return success, frames


def slug(s, n=40):
    """Filesystem-safe short label from an instruction."""
    return "".join(c if c.isalnum() else "_" for c in s)[:n].strip("_")


In [ ]:
num_rollouts_per_task = 20
max_steps = 400


In [ ]:
results = {"suite": {}, "task": {}}

for task in tqdm(all_tasks, desc="Tasks"):
    if trained_instructions and task["language"] not in trained_instructions:
        continue

    bddl = os.path.join(get_libero_path('bddl_files'), task["suite"], task["bddl_file"])
    env = OffScreenRenderEnv(bddl_file_name=bddl, camera_heights=256, camera_widths=256)
    env.seed(42)

    successes = 0
    for r in range(num_rollouts_per_task):
        init = task["init_states"][r % len(task["init_states"])]
        ok, frames = run_rollout(env, init, task["language"], max_steps=max_steps)
        successes += int(ok)

        # Filename encodes suite, task, rollout idx, outcome.
        outcome = "success" if ok else "fail"
        fname = f"{task['suite']}__{slug(task['language'])}__r{r}__{outcome}.mp4"
        path = os.path.join(video_dir, fname)
        vwrite(path, np.stack(frames), inputdict={"-r": "20"})

    env.close()

    suite_results = results["suite"].setdefault(task["suite"], [0, 0])
    suite_results[0] += successes
    suite_results[1] += num_rollouts_per_task

    results["task"][f"{task['suite']}/{task['language']}"] = (
        successes / num_rollouts_per_task
    )
    print(f"{task['suite']}/{task['language']}: "
          f"{successes}/{num_rollouts_per_task}")

print()
print("=== Suite totals ===")
for suite, (s, n) in results["suite"].items():
    print(f"  {suite}: {s}/{n} = {100*s/n:.1f}%")

In [ ]:
from google.colab import drive
import shutil
import glob
import os

drive.mount('/content/drive')

source_pattern = '/content/rollouts/*.mp4'
dest_dir = '/content/drive/MyDrive/DuPLO_Rollouts'

os.makedirs(dest_dir, exist_ok=True)

rollouts = glob.glob(source_pattern)

if not rollouts:
    print("No rollouts found.")
else:
    for video in rollouts:
        file_name = os.path.basename(video)
        dest_path = os.path.join(dest_dir, file_name)
        print(f"Copying {file_name} to Google Drive...")
        shutil.copy2(video, dest_path)
    print("\nAll rollouts successfully copied to Google Drive!")

In [ ]:
!tar -czvf /content/eval_videos.tar.gz -C /content/DuPLO-VLA eval_videos

In [ ]:
import os
import glob
import time
from google.colab import runtime

dest_dir = '/content/drive/MyDrive/DuPLO_Rollouts'
saved_files = glob.glob(os.path.join(dest_dir, '*.mp4'))

if len(saved_files) > 0:
    print(f"Verified {len(saved_files)} rollout video(s) in Google Drive.")
    print("Disconnecting runtime in 5 seconds to save compute resources...")
    time.sleep(5)
    runtime.unassign()
else:
    time.sleep(60)
    runtime.unassign()